# MPP Converter Output Reader

Read CSV output files from Azure Blob Storage using the MPP converter settings in `.env`.

In [1]:
import os
from io import BytesIO
from pathlib import Path

import pandas as pd
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv


REPO_ROOT = Path.cwd()
if not (REPO_ROOT / ".env").exists():
    REPO_ROOT = Path.cwd().parent

load_dotenv(REPO_ROOT / ".env")

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING_MPP_CONVERTER")
container_name = os.getenv("AZURE_STORAGE_CONTAINER_MPP_CONVERTER_OUT")

if not connection_string:
    raise ValueError("AZURE_STORAGE_CONNECTION_STRING_MPP_CONVERTER is not configured in .env")

blob_service = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service.get_container_client(container_name)

print(f"Connected to Azure Blob container: {container_name}")

Connected to Azure Blob container: mppoutputnew


In [2]:
blobs = list(container_client.list_blobs())
csv_blobs = [blob for blob in blobs if blob.name.lower().endswith(".csv")]

pd.DataFrame(
    [
        {
            "name": blob.name,
            "size_bytes": blob.size,
            "last_modified": blob.last_modified,
        }
        for blob in csv_blobs
    ]
)

,name,size_bytes,last_modified
0,Plymouth WP1 schedule final.csv,21409,2026-05-03 16:01:55+00:00
1,Sunderland LEVI.csv,22267,2026-05-03 14:09:57+00:00
2,Surrey WP11 Schedule (new).csv,26322,2026-05-03 16:03:59+00:00
3,Surrey WP12 Schedule1.csv,22244,2026-05-03 14:13:06+00:00
4,Warrington Project Plan.csv,39226,2026-05-03 14:29:03+00:00
5,__Project_Template_CK.csv,27293,2026-05-04 19:29:56+00:00


In [3]:
def read_csv_blob(blob_name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download a CSV blob from the output container and return it as a DataFrame."""
    blob_client = container_client.get_blob_client(blob_name)
    content = blob_client.download_blob().readall()
    return pd.read_csv(BytesIO(content), **read_csv_kwargs)


csv_dataframes = {blob.name: read_csv_blob(blob.name) for blob in csv_blobs}

print(f"Loaded {len(csv_dataframes)} CSV blob(s).")
list(csv_dataframes.keys())

Loaded 6 CSV blob(s).


['Plymouth WP1 schedule final.csv',
 'Sunderland LEVI.csv',
 'Surrey WP11 Schedule (new).csv',
 'Surrey WP12 Schedule1.csv',
 'Warrington Project Plan.csv',
 '__Project_Template_CK.csv']

In [4]:
df=csv_dataframes['Sunderland LEVI.csv']

In [5]:
df

,TaskID,TaskName,StartDate,FinishDate,WeekOfYear
0,0,CK Project Plan - LBH - Car Parks V2,2025-05-12,2026-08-03,32
1,1,Connected Kerb Project Plan - Sunderlan LEVI -...,2025-05-12,2026-08-03,32
2,2,Definition,2025-07-08,2025-12-30,1
3,3,Planning,2025-07-08,2026-01-07,2
4,4,Familiarisation call Sales and SDM [if a new c...,2025-07-08,2025-07-08,28
...,...,...,...,...,...
342,342,20. Site Audits complete,2026-06-15,2026-06-15,25
343,343,Handover Document,2026-07-16,2026-07-22,30
344,344,Charge Point Registration (National Chargepoin...,2026-07-23,2026-07-29,31
345,345,Project handover / sign-off,2026-07-30,2026-07-31,31


In [ ]:
import re
from pathlib import PurePosixPath

OUTPUT_COLUMNS = ["Work Package", "Gate Number", "Year", "Week of Year", "StartDate", "FinishDate"]


def _normalized_column_name(column_name: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(column_name).lower())


def find_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    normalized_columns = {_normalized_column_name(column): column for column in df.columns}
    for candidate in candidates:
        column = normalized_columns.get(_normalized_column_name(candidate))
        if column is not None:
            return column
    return None


def build_gate_table_for_df(df: pd.DataFrame, blob_name: str) -> pd.DataFrame:
    task_col = "TaskName"
    start_col = "StartDate"
    finish_col = "FinishDate"
    work_package_col = blob_name

    required_columns = {
        "task/name": task_col,
        "start date": start_col,
        "finish date": finish_col,
    }
    missing_columns = [name for name, column in required_columns.items() if column is None]
    if missing_columns:
        print(f"Skipping {blob_name}: missing {', '.join(missing_columns)} column(s)")
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    gate_rows = df.copy()
    gate_extract = gate_rows[task_col].astype(str).str.extract(
        r"^\s*(?:\d+\.)?\s*Gate\s+(?P<gate_number>\d+)\s*$",
        flags=re.IGNORECASE,
    )
    gate_rows = gate_rows[gate_extract["gate_number"].notna()].copy()
    gate_rows["Gate Number"] = gate_extract.loc[gate_rows.index, "gate_number"].astype(int)

    start_dates = pd.to_datetime(gate_rows[start_col], errors="coerce")
    finish_dates = pd.to_datetime(gate_rows[finish_col], errors="coerce")
    iso_calendar = start_dates.dt.isocalendar()

    work_package = (
        gate_rows[work_package_col]
        if work_package_col is not None
        else PurePosixPath(blob_name).stem
    )

    return pd.DataFrame(
        {
            "Work Package": work_package,
            "Gate Number": gate_rows["Gate Number"],
            "Year": iso_calendar.year,
            "Week of Year": iso_calendar.week,
            "StartDate": start_dates.dt.date,
            "FinishDate": finish_dates.dt.date,
        }
    )[OUTPUT_COLUMNS]


gate_table = pd.concat(
    [build_gate_table_for_df(df, blob_name) for blob_name, df in csv_dataframes.items()],
    ignore_index=True,
) if csv_dataframes else pd.DataFrame(columns=OUTPUT_COLUMNS)



In [13]:
for blob_name, df in csv_dataframes.items():
    print(f"Processing {blob_name} with columns: {list(df.columns)}")

Processing Plymouth WP1 schedule final.csv with columns: ['TaskID', 'TaskName', 'StartDate', 'FinishDate', 'WeekOfYear']
Processing Sunderland LEVI.csv with columns: ['TaskID', 'TaskName', 'StartDate', 'FinishDate', 'WeekOfYear']
Processing Surrey WP11 Schedule (new).csv with columns: ['TaskID', 'TaskName', 'StartDate', 'FinishDate', 'WeekOfYear']
Processing Surrey WP12 Schedule1.csv with columns: ['TaskID', 'TaskName', 'StartDate', 'FinishDate', 'WeekOfYear']
Processing Warrington Project Plan.csv with columns: ['TaskID', 'TaskName', 'StartDate', 'FinishDate', 'WeekOfYear']
Processing __Project_Template_CK.csv with columns: ['TaskID', 'TaskName', 'StartDate', 'FinishDate', 'WeekOfYear']


In [16]:
df.head()

,TaskID,TaskName,StartDate,FinishDate,WeekOfYear
0,0,CK Project Plan - LBH - Car Parks V2,2025-10-01,2026-06-22,26
1,1,Connected Kerb Project Plan - [Client] [WP X] ...,2025-10-01,2026-06-22,26
2,2,Definition,2025-10-01,2026-03-25,13
3,3,Planning,2025-10-01,2026-01-07,2
4,4,Familiarisation call Sales and SDM [if a new c...,2025-10-01,2025-10-01,40
